**Travelling Salesman Problem using Ant Colony Optimization**

In [ ]:
# cities = nodes
# ants = travellers

import random
from collections import namedtuple
import matplotlib.pyplot as pl0t

# define constant value
MaxN = 1111
val = 1e-6
nAnt = 55
# nAnt = 8   # for small N
# initialPheromone = 0.8
initialPheromone = 0.3
alpha = 1.0
beta = 1.0
# beta = 1.2
P = 0.3
Q = 10
# Q = 1
x = 0
y = 0

INF = 1000000000
Best = INF
Iteration = 2000
# Iteration = 20
# inputfile = "SmallDataset.INP"
inputfile = "Graph.INP"
listofProb = []

c = [[0 for i in range(MaxN)] for j in range(MaxN)]
Pheromone = [[0 for i in range(MaxN)] for j in range(MaxN)]
Delta = [[0 for i in range(MaxN)] for j in range(MaxN)]
Visited = [False for i in range(MaxN)]
Path = [0 for i in range(MaxN+1)]           # list of cities that ant visit
OldPath = [0 for i in range(MaxN)]
Bestpath = [0 for i in range(MaxN)]
StartPoint = [0 for i in range(nAnt)]

# define point structure
class point:
  def __init__(self, x = 0, y = 0):
    self.x = x
    self.y = y

p = [point() for i in range(MaxN)]      # we are initialising/creating a variable of type point

###### form the graph #####
# input format #
# <width> <height> <n>
# <x> <y>
# def graph():
#   global n
#   w, h, n = map(int, input().split())
#   w = int(w)
#   h = int(h)
#   n = int(n)
#   # print(n)

#   # take (x,y) coordinates as input
#   with open("Graph.INP", "w") as graphFile:
#     i = 1
#     graphFile.write(f"{w} {h} {n}\n")
#     print("Inside the file")
#     for i in range(0, n):
#       x = random.randint(1, MaxN)
#       y = random.randint(1, MaxN)
#       graphFile.write(f"{x%w} {y%h}\n")
#       p[i].x = x
#       p[i].y = y

#### Find length ####
def findLength(x1, y1, x2, y2):            # get your euclidean distance between 2 points
  dx = (x2 - x1) ** 2
  dy = (y2 - y1) ** 2
  dist = (dx + dy) ** 0.5
  return dist

def Probability(Pheromone, alpha, Visibility, beta):   # calculate probability
  if Pheromone <= 0 and Visibility <= 0:
    return 0
  return (Pheromone ** alpha) * (Visibility ** beta)

def Visibility(i, j):
  if (i == j) or (c[i][j] == 0) :
    return 0
  else:
    return float(1/ c[i][j])

#### Input function ####
# Intialise the matrix c with distances and define the graph. We are just reading coordinates and height, width from input file and then we are just calc distances bet points.put in matrix

def inputFunc():

  global n, p, c, Pheromone, Delta, StartPoint, Best, Bestpath

  print("### Hyperparameters ###")
  print("T = "+str(Iteration))
  print("InitialPheromone = ", str(initialPheromone))
  print("m = "+str(nAnt))
  print("epsilon = ", str(val))
  print("rho = "+str(P))   # this is evaporation rate val
  print("beta = ", str(beta))
  print("alpha = ", str(alpha))
  print("Q = "+str(Q))     #
  print("")

  with open(inputfile, "r") as graphFile:
    linesread = graphFile.readlines()                           # read first line wifth, height and n = no of coordinates
    # print("Inside input func\n",linesread[0])
    # spliotting the first line width, height and n
    w, h, n = map(int, linesread[0].split())

    # for subsequent lines reading                # get points
    for i in range(0, n):
        x, y = map(int, linesread[i+1].split())
        # print(x, y)
        p[i].x = x
        p[i].y = y

    # we want dist matrix
    for i in range(0, n):                        # get distance between those points picked before
      for j in range(i+1, n):
        dist_val = findLength(p[i].x, p[i].y, p[j].x, p[j].y)  # dist_val here is the euclidean distance
        if dist_val < 0.1:
          dist_val = 0.1
        c[i][j] = dist_val                       # this c matrix which 2D has the euclidean distances that says how far those cities are. c[i][j] means i to j ka dist
        c[j][i] = c[i][j]                        # dist from A to B  = dist from B to A


#### init ####
# Initialise Pheromone and startpoint.
def initFunc():
  global Best
  Best = INF                                    # we need to find out this best solution. So initially this val is infinity

  for i in range(0, n):                         # initialise pheromone matrix with initialpheromone value = very small value of pheromone
    for j in range(0, n):
      Pheromone[i][j] = initialPheromone

  # randomly pick start city numbers and assign to ants
  for i in range(0, nAnt):                      # StartPoint stores index of cities for nAnts here. StartPoint is an array. Eg ant 1 = city 0
    StartPoint[i] = random.randint(0, n-1) % n  # start from starting point value present in the graph (matrix) and then go to every city comebacjk.

# Performed for obtaining the local optimization
def mutation(Path, Current):
  while True:
      stop = True

      for i in range(0, n):
          for j in range(i + 2, n):
              x = Path[i]
              y = Path[(i + 1) % n]
              u = Path[j]
              v = Path[(j + 1) % n]

              if c[x][u] + c[y][v] < c[x][y] + c[u][v]:
                  stop = False

                  # Save current path to OldPath no doing it the other way
                  for t in range(n):
                      OldPath[t] = Path[t]

                  # Perform circular swap here
                  t = i
                  h = j
                  while True:
                      t = (t + 1) % n
                      Path[t] = OldPath[h]
                      if h == (i + 1) % n:
                          break
                      h = (h - 1 + n) % n

                  h = (j + 1) % n
                  while True:
                      if h == 0:
                          break
                      t = (t + 1) % n
                      Path[t] = OldPath[h]
                      h = (h + 1) % n

                  Current += c[x][u] + c[y][v] - c[x][y] - c[u][v]

      if stop:
          break

  return Current


#### FindPath ####
# s = start city
# Here we are identifying the complete tour of the ant from stat city an traverse all the nodes from start city and come back
#### FindPath ####
# s = start city
# Here we are identifying the complete tour of the ant from start city an traverse all the nodes from start city and come back
def FindPath(s):
  global Best, Bestpath

  # for everytime we call this function, we call it for a new ant so we initialise everything as False.
  # visited city = false because nothing is visited yet and we are at start
  for i in range(0, n):
    Visited[i] = False


  # start from start city
  Path[0] = s
  Visited[Path[0]] = True     # bcz 0 is already visited this is the startcity it is visited, its val = true

  for step in range(1, n):
    # u = start from first city
    u = Path[step-1]
    MaxProb = -0.01
    fullProb = 0.0
    next_city = -1
    cand = []
    listofProb = []

   # for every city that we explore we have few posibilities of cities infront of us that we can choose with highest probability.
   # v represets all the cities in the neighbourhood that are unvisited
    for v in range(0, n):                               # v = unvisited cities
      if Visited[v] == False:                           # unvisited
        # compute prob. wehave to choose ccity ith highest prob
        ##########  Pheromone = tao and Visibility = eta
        Prob = Probability(Pheromone[u][v], alpha, Visibility(u, v), beta)
        listofProb.append(Prob)
        fullProb += Prob
        cand.append(v)

    # stoch
    if fullProb > 0:
      cum = 0.0
      r = random.random() * fullProb  # random number between 0 and fullProb
      for idx in range(len(cand)):    # loop through all cand
        cum += listofProb[idx]        # collectprobability like this we add
        if r <= cum:                  # if random number falls in this range
          next_city = cand[idx]       # pick this one city we got
          break
    else:
      # select rand city
      # check len(candidate)
      if len(cand) > 0:
        next_city = random.choice(cand)

    if next_city == -1:                                 # no more cities found to explore
      for v in range(0, n):
        if Visited[v] == False:
          next_city = v
          break

    Path[step] = next_city
    # visited = true when highest prob is found
    Visited[Path[step]] = True

  Path[n] = Path[0]
  # after we get the next city selected calculate length of the tour
  Current = 0
  for i in range(0, n):
    u = Path[i]        # current city
    v = Path[i+1]      # next city  that %n bcz we want to come back to start city so it is wrapping around after last city
    Current += c[u][v]   # total distance value that is found out by the ant.

  Current = mutation(Path, Current)  # Not in ACO but improves path explortion.

  for i in range(0, n):
    u = Path[i]
    v = Path[(i+1)%n]
    if Current > 0:
      Delta[u][v] += Q/Current

  if Current < Best:
    Best = Current
    for i in range(0, n+1):
      Bestpath[i] = Path[i]

#### AntColony ####
# Identify path where we visit everycity
# Update pheromont trail to converge on the best path
def AntColony():
  global Best

  Best = INF
  # each Time here is that all ants reached the dest and phermones are updated
  for Time in range(1, Iteration+1):
    previous_best = Best

    # initial pheromone valu e - 0
    for i in range(0, n):
      for j in range(0, n):

        # Delta = temporary is the new pheromone to be added at specific edge during iteration
        # cleared initially because I want to update this value with the new value after
        # all ants finishes the entire path.
        Delta[i][j] = 0.0

    # find path of ant. every ant starting from a different startpoint
    # delta will get updated with new pheromone
    for i in range(0, nAnt):
      FindPath(StartPoint[i])

    # Pheromone updation step here
    for i in range(0, n):
      for j in range(0, n):
        # P = amt of evaporation (0.4 = 40% phermone evaporates ) ,
        # Delta = new pheromone deposited at present by ants
        Pheromone[i][j] = P * Pheromone[i][j] + Delta[i][j]         # P = amount of pheromone that is left after the pheromone evaported ; Pheromone is new pheromone ants deposit; Delta = newly deposted phero

    print("Iteration "+str(Time)+": Best distance Obtained = "+str(Best))

def output():
  print("Best Path = ")
  for i in range(0, n+1):
    print(Bestpath[i], end = " ")
  print("\nBest Path Length = ", Best)


def visualise():

    global Bestpath, p, n, Best, INF

    if n == 0 or Best == INF:
        print("OptimalPath not found anymore")
        return

    xcoord = []
    ycoord = []
    for i in range(0, n):
        idx = Bestpath[i]       # index bestpath
        xcoord.append(p[idx].x)
        ycoord.append(p[idx].y)

    # wrap around here move to start city again
    start_idx = Bestpath[0]
    xcoord.append(p[start_idx].x)
    ycoord.append(p[start_idx].y)

    pl0t.figure(figsize=(10, 8))
    pl0t.plot(xcoord, ycoord, marker='o', linestyle='-', label='Best tour')

    pl0t.xlabel('X-coordinate')
    pl0t.ylabel('Y-coordinate')
    pl0t.title(f'Travelling Salesman Problem = {Best}')
    pl0t.grid(True)
    pl0t.legend()
    pl0t.show()



#### main ####
def main():
  print("Travelling Salesman Problem using Ant Colony Optimization")
  # graph()
  print(open(inputfile).read())
  inputFunc()
  initFunc()
  AntColony()
  output()
  visualise()

if __name__ == "__main__":
  main()

Travelling Salesman Problem using Ant Colony Optimization
800 600 60
791 467
51 417
253 293
372 528
634 47
251 491
115 68
378 458
657 174
209 180
642 556
328 215
327 93
78 416
114 190
58 202
172 57
702 461
589 225
424 26
70 575
43 428
286 585
711 173
554 295
203 73
88 596
290 331
547 568
310 320
522 13
222 568
381 46
671 276
303 570
145 48
76 573
110 341
514 162
409 221
46 268
217 212
695 335
455 107
427 194
207 565
686 101
748 381
457 467
790 171
19 360
685 487
228 95
9 147
721 143
563 570
582 167
102 180
39 229
23 309

### Hyperparameters ###
T = 2000
InitialPheromone =  0.3
m = 55
epsilon =  1e-06
rho = 0.3
beta =  1.0
alpha =  1.0
Q = 10

Iteration 1: Best distance Obtained = 4107.618936475412
Iteration 2: Best distance Obtained = 4097.78714750565
Iteration 3: Best distance Obtained = 4086.519122830458
Iteration 4: Best distance Obtained = 4086.519122830458
Iteration 5: Best distance Obtained = 4076.68733386069
Iteration 6: Best distance Obtained = 4076.68733386069
Iteration 7: Bes